In [3]:
# 02_Train_Test_Split.py
import pandas as pd
from sklearn.model_selection import train_test_split
import os
import json

INPUT_PATH = r"C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\intermediate\encoded"
OUTPUT_PATH = r"C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\intermediate\splited"

df = pd.read_parquet(f"{INPUT_PATH}/df_triage_encoded.parquet")

X = df.drop(columns=["nivel_triage"])
y = df["nivel_triage"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Guardar archivos
os.makedirs(OUTPUT_PATH, exist_ok=True)
X_train.to_parquet(f"{OUTPUT_PATH}/X_train.parquet")
X_test.to_parquet(f"{OUTPUT_PATH}/X_test.parquet")
y_train.to_frame().to_parquet(f"{OUTPUT_PATH}/y_train.parquet")
y_test.to_frame().to_parquet(f"{OUTPUT_PATH}/y_test.parquet")

# Calcular métricas descriptivas
total_samples = len(df)
n_features = X.shape[1]

class_dist_total = y.value_counts(normalize=True).mul(100).round(2).to_dict()
class_dist_train = y_train.value_counts(normalize=True).mul(100).round(2).to_dict()
class_dist_test = y_test.value_counts(normalize=True).mul(100).round(2).to_dict()

metrics = {
    "total_samples": total_samples,
    "n_features": n_features,
    "class_distribution_percent": class_dist_total,
    "train_distribution_percent": class_dist_train,
    "test_distribution_percent": class_dist_test
}

# Guardar métricas en JSON
with open(os.path.join(OUTPUT_PATH, "split_metrics.json"), "w") as f:
    json.dump(metrics, f, indent=4)

# Mostrar en consola
print(f"\n📈 Métricas del split:")
print(f"🧾 Total de muestras: {total_samples}")
print(f"🧮 Total de features: {n_features}")
print("\n📊 Distribución de clases (%):")
print("  Clase | Total | Train | Test")
print("  ------|-------|-------|------")
for c in sorted(y.unique()):
    total = class_dist_total.get(c, 0)
    train = class_dist_train.get(c, 0)
    test = class_dist_test.get(c, 0)
    print(f"   {c:<5} | {total:>5.1f}% | {train:>5.1f}% | {test:>5.1f}%")

print(f"\n💾 Métricas guardadas en: split_metrics.json")



📈 Métricas del split:
🧾 Total de muestras: 560486
🧮 Total de features: 539

📊 Distribución de clases (%):
  Clase | Total | Train | Test
  ------|-------|-------|------
   1     |   9.3% |   9.3% |   9.3%
   2     |  19.4% |  19.4% |  19.4%
   3     |  21.3% |  21.3% |  21.3%
   4     |  22.3% |  22.3% |  22.3%
   5     |  27.7% |  27.7% |  27.7%

💾 Métricas guardadas en: split_metrics.json
